# Fresh-runtime reproduction: Gaussian denoiser + scaled DPAR
Этот notebook предназначен для чистого runtime. Он клонирует репозиторий, запускает тесты, заново обучает Gaussian denoiser, выполняет calibration и frozen held-out follow-up. Рекомендуется GPU.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if repo.exists(): subprocess.run(['rm','-rf',str(repo)], check=True)
subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
print('cwd:', os.getcwd())


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print('gpu:', torch.cuda.get_device_name(0))


## Быстрые проверки воспроизводимости кода


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_inference_followups.py','tests/test_denoiser.py'], check=True)


## Полное воспроизведение с чистого runtime
Pipeline восстанавливает frozen sentiment direction при необходимости, выполняет real-data preflight, кэширует generic layer-6 activations, переобучает Gaussian denoiser, проверяет validation MSE improvement, калибрует `beta` и один раз оценивает frozen held-out prompts/seeds.


In [ ]:
subprocess.run([sys.executable,'scripts/run_retrain_gaussian_followups.py','--config','configs/retrain_gaussian_followups_gpt2.yaml'], check=True)


## Проверка воспроизведённых научных результатов


In [ ]:
import json, pandas as pd
from pathlib import Path
history = json.loads(Path('results/retrained_denoiser_gaussian_history.json').read_text())
print('final validation improvement:', history[-1]['val_relative_mse_improvement'])
print(Path('results/retrained_inference_followups/SUMMARY.md').read_text())
display(pd.read_csv('results/retrained_inference_followups/heldout_interpolated_frontier.csv'))


Reference: final Gaussian validation relative MSE improvement ≈ **0.6781128742**. Небольшие generation differences возможны при изменениях upstream libraries; prompts, seeds, model name, hook, training seed и evaluation protocol зафиксированы.
